### Build clean training text

In [1]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("afaskar/Aqar.fm-Saudi-Real-Estate-Listings", "default")
df = pd.DataFrame(ds["train"])
df = df.fillna("")

def bool_ar(x, yes="نعم", no="لا"):
    if x is True:
        return yes
    if x is False:
        return no
    return ""

def row_to_text(row):
    parts = []

    if row["category_name"]:
        parts.append(f'{row["category_name"]}')
    if row["district"] and row["city"]:
        parts.append(f'في {row["district"]} بمدينة {row["city"]}.')
    elif row["city"]:
        parts.append(f'في مدينة {row["city"]}.')
    
    if row["price"]:
        parts.append(f'السعر {int(row["price"])} ريال.')
    if row["area_sqm"]:
        parts.append(f'المساحة {row["area_sqm"]} متر مربع.')
    if row["num_bedrooms"]:
        parts.append(f'عدد غرف النوم {int(row["num_bedrooms"])}.')
    if row["num_bathrooms"]:
        parts.append(f'عدد دورات المياه {int(row["num_bathrooms"])}.')
    if row["num_living_rooms"]:
        parts.append(f'عدد غرف المعيشة {int(row["num_living_rooms"])}.')
    
    features = []
    if row["furnished"] is not "":
        features.append(f'مفروش: {bool_ar(row["furnished"])}')
    if row["ac"] is not "":
        features.append(f'مكيف: {bool_ar(row["ac"])}')
    if row["lift"] is not "":
        features.append(f'مصعد: {bool_ar(row["lift"])}')
    if row["car_entrance"] is not "":
        features.append(f'مدخل سيارة: {bool_ar(row["car_entrance"])}')
    if row["pool"] is not "":
        features.append(f'مسبح: {bool_ar(row["pool"])}')
    
    if features:
        parts.append("المميزات: " + "، ".join(features) + ".")
    
    if row["description"]:
        parts.append("الوصف: " + str(row["description"]).strip())

    return " ".join(parts)

<>:37: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:39: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:41: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:43: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:45: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:37: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:39: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:41: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:43: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:45: SyntaxWarning: "is not" with a literal. Did you mean "!="?
C:\Users\naifa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_t

In [2]:
texts = df.apply(row_to_text, axis=1).tolist()

print(texts[0])
print("Number of samples:", len(texts))

دور للبيع في حي النرجس بمدينة الرياض. السعر 1500000 ريال. المساحة 116.0 متر مربع. عدد غرف النوم 4. عدد دورات المياه 3. عدد غرف المعيشة 1. المميزات: مفروش: لا، مكيف: لا، مصعد: لا، مدخل سيارة: نعم، مسبح: لا. الوصف: مشروع ادوار ســـدرا مشروع يتميز بموقعه الاستراتجي بحي النرجس جنوب طريق انس بن مالك وتصاميمه الفريده. دور ثاني بمساحه 169م محتويات الدور: -غرفه نوم ماستر -غرفه نوم 1 -غرفه نوم 2 -غرفه خادمه بدوره مياه -صاله -مطبخ -سطح امامي -بلكونه عند الغرفه الماستر -3 دورات مياه
Number of samples: 6252


### Save pretraining file

In [3]:
import os

os.makedirs("../data/pretrain", exist_ok=True)

with open("../data/pretrain/data.txt", "w", encoding="utf-8") as f:
    for t in texts:
        f.write(t.strip() + "\n")

### summarize listing

In [4]:
def make_summary_output(row):
    parts = []
    if row["category_name"]:
        parts.append(f'{row["category_name"]}')
    if row["district"] and row["city"]:
        parts.append(f'في {row["district"]} بمدينة {row["city"]}')
    elif row["city"]:
        parts.append(f'في {row["city"]}')
    if row["price"]:
        parts.append(f'بسعر {int(row["price"])} ريال')
    if row["area_sqm"]:
        parts.append(f'بمساحة {row["area_sqm"]} متر مربع')
    if row["num_bedrooms"]:
        parts.append(f'ويحتوي على {int(row["num_bedrooms"])} غرف نوم')
    if row["num_bathrooms"]:
        parts.append(f'و{int(row["num_bathrooms"])} دورات مياه')
    return "، ".join(parts) + "."

### answer property facts

In [5]:
def make_fact_output(row):
    answers = []
    if row["category_name"]:
        answers.append(f'نوع العقار: {row["category_name"]}')
    if row["city"]:
        answers.append(f'المدينة: {row["city"]}')
    if row["district"]:
        answers.append(f'الحي: {row["district"]}')
    if row["price"]:
        answers.append(f'السعر: {int(row["price"])} ريال')
    if row["area_sqm"]:
        answers.append(f'المساحة: {row["area_sqm"]} متر مربع')
    return "، ".join(answers) + "."

In [6]:
sft_data = []

for _, row in df.iterrows():
    listing_text = row_to_text(row)

    sft_data.append({
        "instruction": "لخص الإعلان العقاري في سطر واحد.",
        "input": listing_text,
        "output": make_summary_output(row)
    })

    sft_data.append({
        "instruction": "استخرج أهم معلومات العقار.",
        "input": listing_text,
        "output": make_fact_output(row)
    })

In [7]:
import json
os.makedirs("../data/finetune", exist_ok=True)

with open("../data/finetune/real_estate_sft.json", "w", encoding="utf-8") as f:
    json.dump(sft_data, f, ensure_ascii=False, indent=2)